In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.fft import fft
from sklearn.preprocessing import StandardScaler

print('All libraries imported successfully')

All libraries imported successfully


In [2]:
def load_data():
    print('Loading UCI HAR Dataset...')
    data_folder = 'UCI HAR Dataset/'

    features  = pd.read_csv(data_folder + 'features.txt', sep='\\s+', header=None, names=['id', 'name'])
    columnNames = features['name'].values
    labelsMap = pd.read_csv(data_folder + 'activity_labels.txt', sep='\\s+', header=None, names=['id', 'activity_name'])

    def read_set(set_name):
        X = pd.read_csv(f'{data_folder}{set_name}/X_{set_name}.txt', sep='\\s+', header=None)
        X.columns = columnNames
        y = pd.read_csv(f'{data_folder}{set_name}/y_{set_name}.txt', sep='\\s+', header=None, names=['Activity_ID'])
        y['Activity_Name'] = y['Activity_ID'].map(labelsMap.set_index('id')['activity_name'])
        subject = pd.read_csv(f'{data_folder}{set_name}/subject_{set_name}.txt', sep='\\s+', header=None, names=['Subject_ID'])
        return pd.concat([subject, y, X], axis=1)

    train_df = read_set('train')
    test_df  = read_set('test')
    print(f'Train shape : {train_df.shape}')
    print(f'Test  shape : {test_df.shape}')
    return train_df, test_df

train_data, test_data = load_data()
train_data.head(3)

Loading UCI HAR Dataset...
Train shape : (7352, 564)
Test  shape : (2947, 564)


,Subject_ID,Activity_ID,Activity_Name,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,...,fBodyBodyGyroJerkMag-meanFreq(),fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)"
0,1,5,STANDING,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,...,-0.074323,-0.298676,-0.710304,-0.112754,0.030400,-0.464761,-0.018446,-0.841247,0.179941,-0.058627
1,1,5,STANDING,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,...,0.158075,-0.595051,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317
2,1,5,STANDING,0.279653,-0.019467,-0.113462,-0.995380,-0.967187,-0.978944,-0.996520,...,0.414503,-0.390748,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118


In [3]:
def load_inertial_signals(set_name):
    print(f'Loading raw signals for {set_name}...')
    data_folder = f'UCI HAR Dataset/{set_name}/Inertial Signals/'
    filenames = [
        'body_acc_x_', 'body_acc_y_', 'body_acc_z_',
        'body_gyro_x_', 'body_gyro_y_', 'body_gyro_z_',
        'total_acc_x_', 'total_acc_y_', 'total_acc_z_'
    ]
    signals = []
    for name in filenames:
        data = pd.read_csv(f'{data_folder}{name}{set_name}.txt', sep='\\s+', header=None)
        signals.append(data.values)
    return np.transpose(np.array(signals), (1, 2, 0))

X_train_raw = load_inertial_signals('train')
X_test_raw  = load_inertial_signals('test')

print(f'X_train_raw shape : {X_train_raw.shape}  -> (samples, timesteps, signals)')
print(f'X_test_raw  shape : {X_test_raw.shape}')

Loading raw signals for train...
Loading raw signals for test...
X_train_raw shape : (7352, 128, 9)  -> (samples, timesteps, signals)
X_test_raw  shape : (2947, 128, 9)


In [4]:
print('=== Data Quality Report ===')
print(f'NaN values in train  : {train_data.isnull().sum().sum()}')
print(f'NaN values in test   : {test_data.isnull().sum().sum()}')
print(f'Duplicate rows       : {train_data.duplicated().sum()}')
print()
print('=== Train/Test Split (already provided by dataset) ===')
print(f'Train samples : {train_data.shape[0]}')
print(f'Test  samples : {test_data.shape[0]}')
print(f'Split ratio   : {train_data.shape[0] / (train_data.shape[0] + test_data.shape[0]):.0%} train / {test_data.shape[0] / (train_data.shape[0] + test_data.shape[0]):.0%} test')
print()
print('=== Class Distribution (train) ===')
print(train_data['Activity_Name'].value_counts())

=== Data Quality Report ===
NaN values in train  : 0
NaN values in test   : 0
Duplicate rows       : 0

=== Train/Test Split (already provided by dataset) ===
Train samples : 7352
Test  samples : 2947
Split ratio   : 71% train / 29% test

=== Class Distribution (train) ===
Activity_Name
LAYING                1407
STANDING              1374
SITTING               1286
WALKING               1226
WALKING_UPSTAIRS      1073
WALKING_DOWNSTAIRS     986
Name: count, dtype: int64


In [5]:
# The 9 sensor signals available in the Inertial Signals folder
SIGNAL_NAMES = [
    'body_acc_x',  'body_acc_y',  'body_acc_z',
    'body_gyro_x', 'body_gyro_y', 'body_gyro_z',
    'total_acc_x', 'total_acc_y', 'total_acc_z'
]

print(f'Number of signals : {len(SIGNAL_NAMES)}')
print(f'Signal names      : {SIGNAL_NAMES}')

Number of signals : 9
Signal names      : ['body_acc_x', 'body_acc_y', 'body_acc_z', 'body_gyro_x', 'body_gyro_y', 'body_gyro_z', 'total_acc_x', 'total_acc_y', 'total_acc_z']


In [ ]:
def extract_time_domain_features(raw_data):
    print('Extracting time-domain features...')
    records = []

    for sample in raw_data:
      
        row = {}
        for i, sig_name in enumerate(SIGNAL_NAMES):
            signal = sample[:, i]                          
            row[f'{sig_name}_mean']     = np.mean(signal)
            row[f'{sig_name}_std']      = np.std(signal)
            row[f'{sig_name}_min']      = np.min(signal)
            row[f'{sig_name}_max']      = np.max(signal)
            row[f'{sig_name}_median']   = np.median(signal)
            row[f'{sig_name}_skew']     = skew(signal)
            row[f'{sig_name}_kurtosis'] = kurtosis(signal)
        records.append(row)

    df = pd.DataFrame(records)
    print(f'  Output shape : {df.shape}  -> {df.shape[0]} samples x {df.shape[1]} features')
    return df


X_train_time = extract_time_domain_features(X_train_raw)
X_test_time  = extract_time_domain_features(X_test_raw)

print()
print('Sample of time-domain features:')
X_train_time.head(3)

Extracting time-domain features...
  Output shape : (7352, 63)  -> 7352 samples x 63 features
Extracting time-domain features...
  Output shape : (2947, 63)  -> 2947 samples x 63 features

Sample of time-domain features:


,body_acc_x_mean,body_acc_x_std,body_acc_x_min,body_acc_x_max,body_acc_x_median,body_acc_x_skew,body_acc_x_kurtosis,body_acc_y_mean,body_acc_y_std,body_acc_y_min,...,total_acc_y_median,total_acc_y_skew,total_acc_y_kurtosis,total_acc_z_mean,total_acc_z_std,total_acc_z_min,total_acc_z_max,total_acc_z_median,total_acc_z_skew,total_acc_z_kurtosis
0,0.002269,0.002941,-0.004294,0.010810,0.002025,0.481111,-0.395797,-0.001655,0.004601,-0.012610,...,-0.124302,-0.034474,-0.408672,0.099732,0.003970,0.088742,0.109485,0.099841,0.071125,0.493800
1,0.000174,0.001981,-0.006706,0.005251,0.000110,-0.480776,1.472747,0.000382,0.005930,-0.012610,...,-0.123188,0.488232,0.163722,0.096939,0.004918,0.081100,0.105788,0.097748,-1.084209,1.257869
2,0.000428,0.002908,-0.010483,0.008167,0.000627,-0.565241,1.802622,-0.001222,0.007310,-0.015333,...,-0.125918,0.366785,-0.640148,0.092897,0.006145,0.081100,0.105788,0.093636,-0.144007,-0.716645


In [ ]:
def spectral_entropy(signal):
    
    fft_vals  = np.abs(fft(signal))[:len(signal)//2]
    power     = fft_vals ** 2
    total     = np.sum(power)
    if total == 0:
        return 0.0
    prob = power / total
    prob = prob[prob > 0]           
    return -np.sum(prob * np.log2(prob))


def extract_fft_features(raw_data, sampling_rate=50):
    print('Extracting FFT frequency-domain features...')
    n_timesteps = raw_data.shape[1]                           
    freqs       = np.fft.fftfreq(n_timesteps, d=1.0/sampling_rate)[:n_timesteps//2]

    records = []
    for sample in raw_data:
        row = {}
        for i, sig_name in enumerate(SIGNAL_NAMES):
            signal   = sample[:, i]
            fft_vals = np.abs(fft(signal))[:n_timesteps//2]

            row[f'{sig_name}_fft_dominant_freq']    = freqs[np.argmax(fft_vals)]
            row[f'{sig_name}_fft_energy']           = np.sum(fft_vals ** 2)
            row[f'{sig_name}_fft_spectral_entropy'] = spectral_entropy(signal)
        records.append(row)

    df = pd.DataFrame(records)
    print(f'  Output shape : {df.shape}  -> {df.shape[0]} samples x {df.shape[1]} features')
    return df


X_train_fft = extract_fft_features(X_train_raw)
X_test_fft  = extract_fft_features(X_test_raw)

print()
print('Sample of FFT features:')
X_train_fft.head(3)

Extracting FFT frequency-domain features...
  Output shape : (7352, 27)  -> 7352 samples x 27 features
Extracting FFT frequency-domain features...
  Output shape : (2947, 27)  -> 2947 samples x 27 features

Sample of FFT features:


,body_acc_x_fft_dominant_freq,body_acc_x_fft_energy,body_acc_x_fft_spectral_entropy,body_acc_y_fft_dominant_freq,body_acc_y_fft_energy,body_acc_y_fft_spectral_entropy,body_acc_z_fft_dominant_freq,body_acc_z_fft_energy,body_acc_z_fft_spectral_entropy,body_gyro_x_fft_dominant_freq,...,body_gyro_z_fft_spectral_entropy,total_acc_x_fft_dominant_freq,total_acc_x_fft_energy,total_acc_x_fft_spectral_entropy,total_acc_y_fft_dominant_freq,total_acc_y_fft_energy,total_acc_y_fft_spectral_entropy,total_acc_z_fft_dominant_freq,total_acc_z_fft_energy,total_acc_z_fft_spectral_entropy
0,0.00000,0.155185,2.610402,0.390625,0.218277,3.594107,0.390625,3.886445,2.556959,0.000000,...,1.344905,0.0,17022.704179,0.000040,0.0,253.073153,0.006050,0.0,163.091410,0.012904
1,7.03125,0.032640,4.761714,0.390625,0.290422,3.365968,0.000000,1.252940,2.680912,0.000000,...,2.333451,0.0,17027.498428,0.000045,0.0,247.813017,0.012676,0.0,154.162921,0.019843
2,0.78125,0.072294,4.779093,0.390625,0.462221,2.989540,0.390625,0.424719,3.435523,0.390625,...,3.248716,0.0,17043.526416,0.000098,0.0,256.328623,0.019163,0.0,141.700207,0.030712


In [8]:
X_train_feats = pd.concat([X_train_time, X_train_fft], axis=1)
X_test_feats  = pd.concat([X_test_time,  X_test_fft],  axis=1)

print('=== Combined Feature Matrix ===')
print(f'X_train_feats shape : {X_train_feats.shape}')
print(f'X_test_feats  shape : {X_test_feats.shape}')
print(f'Total features      : {X_train_feats.shape[1]}')
print(f'  - Time-domain     : {X_train_time.shape[1]}')
print(f'  - FFT             : {X_train_fft.shape[1]}')
print()
X_train_feats.head(3)

=== Combined Feature Matrix ===
X_train_feats shape : (7352, 90)
X_test_feats  shape : (2947, 90)
Total features      : 90
  - Time-domain     : 63
  - FFT             : 27



,body_acc_x_mean,body_acc_x_std,body_acc_x_min,body_acc_x_max,body_acc_x_median,body_acc_x_skew,body_acc_x_kurtosis,body_acc_y_mean,body_acc_y_std,body_acc_y_min,...,body_gyro_z_fft_spectral_entropy,total_acc_x_fft_dominant_freq,total_acc_x_fft_energy,total_acc_x_fft_spectral_entropy,total_acc_y_fft_dominant_freq,total_acc_y_fft_energy,total_acc_y_fft_spectral_entropy,total_acc_z_fft_dominant_freq,total_acc_z_fft_energy,total_acc_z_fft_spectral_entropy
0,0.002269,0.002941,-0.004294,0.010810,0.002025,0.481111,-0.395797,-0.001655,0.004601,-0.012610,...,1.344905,0.0,17022.704179,0.000040,0.0,253.073153,0.006050,0.0,163.091410,0.012904
1,0.000174,0.001981,-0.006706,0.005251,0.000110,-0.480776,1.472747,0.000382,0.005930,-0.012610,...,2.333451,0.0,17027.498428,0.000045,0.0,247.813017,0.012676,0.0,154.162921,0.019843
2,0.000428,0.002908,-0.010483,0.008167,0.000627,-0.565241,1.802622,-0.001222,0.007310,-0.015333,...,3.248716,0.0,17043.526416,0.000098,0.0,256.328623,0.019163,0.0,141.700207,0.030712


In [ ]:
scaler = StandardScaler()

X_train_scaled_arr = scaler.fit_transform(X_train_feats)

X_test_scaled_arr  = scaler.transform(X_test_feats)


X_train_scaled = pd.DataFrame(X_train_scaled_arr, columns=X_train_feats.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled_arr,  columns=X_test_feats.columns)

print('=== Scaling Complete ===')
print(f'X_train_scaled shape : {X_train_scaled.shape}')
print(f'X_test_scaled  shape : {X_test_scaled.shape}')
print()
print('Verify scaling (train should be ~0 mean, ~1 std):')
print(X_train_scaled.describe().loc[['mean','std']].round(3).iloc[:, :5])

=== Scaling Complete ===
X_train_scaled shape : (7352, 90)
X_test_scaled  shape : (2947, 90)

Verify scaling (train should be ~0 mean, ~1 std):
      body_acc_x_mean  body_acc_x_std  body_acc_x_min  body_acc_x_max  \
mean              0.0             0.0             0.0            -0.0   
std               1.0             1.0             1.0             1.0   

      body_acc_x_median  
mean               -0.0  
std                 1.0  


In [ ]:
train_labels = train_data[['Subject_ID', 'Activity_ID', 'Activity_Name']].reset_index(drop=True)
test_labels  = test_data[['Subject_ID', 'Activity_ID', 'Activity_Name']].reset_index(drop=True)

train_final = pd.concat([train_labels, X_train_scaled.reset_index(drop=True)], axis=1)
test_final  = pd.concat([test_labels,  X_test_scaled.reset_index(drop=True)],  axis=1)

train_final.to_csv('train_features.csv', index=False)
test_final.to_csv('test_features.csv',   index=False)

print('Files saved:')
print('  train_features.csv')
print('  test_features.csv')
print()
print(f'train_final shape : {train_final.shape}')
print(f'test_final  shape : {test_final.shape}')
train_final.head(3)

Files saved:
  train_features.csv
  test_features.csv

train_final shape : (7352, 93)
test_final  shape : (2947, 93)


,Subject_ID,Activity_ID,Activity_Name,body_acc_x_mean,body_acc_x_std,body_acc_x_min,body_acc_x_max,body_acc_x_median,body_acc_x_skew,body_acc_x_kurtosis,...,body_gyro_z_fft_spectral_entropy,total_acc_x_fft_dominant_freq,total_acc_x_fft_energy,total_acc_x_fft_spectral_entropy,total_acc_y_fft_dominant_freq,total_acc_y_fft_energy,total_acc_y_fft_spectral_entropy,total_acc_z_fft_dominant_freq,total_acc_z_fft_energy,total_acc_z_fft_spectral_entropy
0,1,5,STANDING,0.200642,-0.868814,0.906009,-0.856036,0.587177,0.383365,-0.331063,...,-1.676605,-0.04378,0.610645,-0.550501,-0.129234,-0.587056,-0.818857,-0.274773,-0.566725,-0.812263
1,1,5,STANDING,0.055948,-0.875426,0.895998,-0.871359,0.544547,-1.181161,0.631206,...,-0.703240,-0.04378,0.611394,-0.550488,-0.129234,-0.588493,-0.813157,-0.274773,-0.569294,-0.807098
2,1,5,STANDING,0.073515,-0.869039,0.880323,-0.863323,0.556060,-1.318546,0.801087,...,0.197969,-0.04378,0.613897,-0.550355,-0.129234,-0.586166,-0.807577,-0.274773,-0.572879,-0.799007


In [12]:
feature_cols = X_train_scaled.columns.tolist()

X_train = train_final[feature_cols]
y_train = train_final['Activity_ID']

X_test  = test_final[feature_cols]
y_test  = test_final['Activity_ID']
print('Activity label mapping:')
print(train_final.groupby(['Activity_ID','Activity_Name']).size().reset_index(name='count'))

Activity label mapping:
   Activity_ID       Activity_Name  count
0            1             WALKING   1226
1            2    WALKING_UPSTAIRS   1073
2            3  WALKING_DOWNSTAIRS    986
3            4             SITTING   1286
4            5            STANDING   1374
5            6              LAYING   1407
